In [4]:
pip install nibabel

Note: you may need to restart the kernel to use updated packages.


In [5]:
import nibabel as nib
import numpy as np
import scipy.io as sio
from collections import defaultdict
from scipy.stats import norm
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt
import os

AttributeError: `np.sctypes` was removed in the NumPy 2.0 release. Access dtypes explicitly instead.

In [7]:
# gRAICAR_output_path = '/Users/fei/Desktop/Harris_Lab/dFNC/CIMT_data/gRAICAR/CIMT_All/output' # path to CIMT output
gRAICAR_output_path = '/Users/fei/Desktop/Harris_Lab/dFNC/FPI_RS_share/gRAICAR/FPI_All/output' # path to FPI output

comp_maps_dir = os.path.join(gRAICAR_output_path, 'compMaps')
comp_maps = sorted(os.listdir(comp_maps_dir))
num_comps = sum(1 for map in comp_maps if map[0] == 'c')
num_comps

NameError: name 'os' is not defined

In [40]:
# Find voxel value percentiles for each aligned component
all_voxels = []

print("5th to 100th Percentiles of Voxel Values, Increments of 5")
for i in range(num_comps):
    img = nib.load(os.path.join(comp_maps_dir, comp_maps[i]))
    data = img.get_fdata()
    flattened_data = data.flatten()
    all_voxels = all_voxels + list(flattened_data)
    print(f"AC {i+1} SD: {np.std(flattened_data)}")
    percs = []
    for j in range(5, 101, 5):
        percs.append(np.percentile(flattened_data, j))
    print(f"Component {i+1}: {percs}, [{np.min(data)}, {np.max(data)}]")

5th to 100th Percentiles of Voxel Values, Increments of 5 (FPI PID1)
AC 1 SD: 7.411665836702054e-06
Component 1: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.029315322009232e-05], [-0.00031906789974112115, 6.029315322009232e-05], 1.118740411483658e-18
AC 2 SD: 7.411665857078771e-06
Component 2: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.5126717758515724e-05], [-0.00034529238631542114, 3.5126717758515724e-05], -6.108907137080288e-16
AC 3 SD: 7.411665851575249e-06
Component 3: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0003194825837722348], [-3.720863285383569e-05, 0.0003194825837722348], 4.743929090340303e-16
AC 4 SD: 7.411665853548544e-06
Component 4: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.000307381161915643], [-3.7911853789318606e-05, 0.000307381161915643], 5.7674760

In [7]:
good_comps = [3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26, 28, 31] # FPI

os.mkdir(os.path.join(gRAICAR_output_path, 'z_scored_compMaps'))
z_scored_path = os.path.join(gRAICAR_output_path, 'z_scored_compMaps')

for i in range(num_comps):
    if i+1 in good_comps: 
        img = nib.load(os.path.join(comp_maps_dir, comp_maps[i]))
        data = img.get_fdata()
        header = img.header
        affine = img.affine 
        flattened_data = data.flatten()
    
        mean = np.mean(flattened_data)
        std = np.std(flattened_data)
    
        z_scores = (data - mean) / std
    
        z_img = nib.Nifti1Image(z_scores, affine, header)
    
        name_split = comp_maps[i].split('.')
        new_name = "_z_scored.".join(name_split)
        nib.save(z_img, os.path.join(z_scored_path, new_name))

In [9]:
# Identify voxels worth keeping
# flatten data then binarize to make mask!
# multiply mask by original maps

z_scored_maps = sorted(os.listdir(z_scored_path))
print(len(z_scored_maps))
good_voxels = set()

for map in z_scored_maps:
    img = nib.load(os.path.join(z_scored_path, map))
    data = img.get_fdata()
    for index, value in np.ndenumerate(data):
        if value > 5.2:
            good_voxels.add(index)
                
print(len(good_voxels))

23
7515


In [13]:
# create mask including only the good voxels
sample_img = nib.load('/Users/fei/Desktop/Harris_Lab/dFNC/FPI_RS_share/PID1_group/r13_01fa_PID1/filtered_func_data.nii')
sample_data = sample_img.get_fdata()
sample_header = sample_img.header
sample_affine = sample_img.affine

mask = np.zeros(sample_data.shape[:3])

for index in good_voxels:
    mask[index] = 1

mask = np.repeat(mask[:, :, :, np.newaxis], sample_data.shape[3], axis=3)

print(mask.shape)
print(np.sum(mask) == len(good_voxels) * sample_data.shape[3])

(96, 96, 25, 450)
True


In [15]:
# save new mask
ica_mask = nib.Nifti1Image(mask, affine=sample_affine, header=sample_header)
nib.save(ica_mask, os.path.join(gRAICAR_output_path, 'ICA_mask.nii'))

In [ ]:
# go to the working directory and run this in terminal
"""
mask="/Users/fei/Desktop/Harris_Lab/dFNC/FPI_RS_share/gRAICAR/FPI_All/output/ICA_mask_5.2.nii"
for folder in */; do
  if [[ "$folder" == PID*_group/ ]]; then
    cd "$folder"
    for subject in */; do
      cd "$subject"
      current=$(basename "$PWD")
      mkdir -p "../../binarized_data_5.2/${current}"
      fslmaths *.nii -mul "$mask" "../../binarized_data_5.2/${current}/${current}_bin.nii"
      cd ..
    done
    cd ..
  fi
done
"""

In [ ]:
# to unzip .nii.gz files
"""
for folder in */; do
    cd "$folder"
    gunzip *.nii.gz
    cd ..
done
"""

In [58]:
comp_arrays = []

for i in range(num_comps):
    img = nib.load(os.path.join(z_scored_compMaps_path, z_scored_maps[i]))
    data = img.get_fdata()
    values = []
    for index, value in np.ndenumerate(data):
        if index in good_voxels:
            values.append(value)
    comp_arrays.append(values)

In [2]:
# Load atlas

atlas_file = '/Users/fei/Desktop/Harris_Lab/dFNC/FPI_RS_share/Template/01_study_specific_atlas_relabel.nii'
atlas = nib.load(atlas_file)
atlas_data = atlas.get_fdata()
print(f"Atlas dimensions: {atlas_data.shape}")

Atlas dimensions: (96, 96, 25)


In [3]:
# Check number of ROIs

roi_names = []
with open('/Users/fei/Desktop/Harris_Lab/dFNC/FPI_RS_share/FPI_labels_not_numbered.txt', 'r') as rois_labels:
    for label in rois_labels:
        roi_names.append(label)

print(f"{int(np.max(atlas_data))} ROIs")
print(f"Length of roi_names equals max value of atlas_data: {len(roi_names) == np.max(atlas_data)}")

121 ROIs
Length of roi_names equals max value of atlas_data: True


In [4]:
# Load gRAICAR results
gRAICAR_output_path = '/Users/fei/Desktop/Harris_Lab/dFNC/gRAICAR/graicar20components/graicar20cd7'
result_mat_file = '20cd7_result.mat'

gRAICAR_result = sio.loadmat(os.path.join(gRAICAR_output_path, result_mat_file))

In [5]:
gRAICAR_result.keys()

dict_keys(['__header__', '__version__', '__globals__', 'obj'])

In [6]:
# Locate "confidence of subject load" values

conf_subj_load = gRAICAR_result['obj'][0, 0][1][0][0][10]
print(conf_subj_load)

[[0.9145 0.5865 0.8205 0.3825 0.5625 0.2725 0.3355 0.044  0.3535 0.1075
  0.18   0.207  0.2545 0.056  0.046  0.06   0.025  0.009  0.001 ]
 [0.8065 0.451  0.679  0.28   0.7555 0.227  0.399  0.0655 0.408  0.135
  0.2475 0.136  0.0705 0.118  0.073  0.099  0.004  0.009  0.006 ]
 [0.723  0.845  0.79   0.713  0.7485 0.6785 0.411  0.4245 0.0635 0.345
  0.27   0.227  0.422  0.4195 0.112  0.0335 0.003  0.005  0.072 ]
 [0.5455 0.013  0.589  0.525  0.138  0.087  0.564  0.109  0.302  0.226
  0.159  0.015  0.088  0.07   0.256  0.212  0.007  0.007  0.007 ]
 [0.924  0.804  0.226  0.7935 0.1315 0.294  0.294  0.3745 0.0685 0.203
  0.207  0.114  0.097  0.1405 0.0995 0.2115 0.002  0.001  0.001 ]
 [0.609  0.942  0.3975 0.705  0.0775 0.322  0.3595 0.108  0.538  0.106
  0.136  0.154  0.062  0.0805 0.093  0.106  0.001  0.003  0.001 ]
 [0.876  0.838  0.6755 0.661  0.8165 0.4175 0.315  0.2555 0.3965 0.187
  0.148  0.474  0.285  0.192  0.012  0.089  0.005  0.001  0.02  ]
 [0.7785 0.7215 0.7335 0.681  0.524  0.4

In [7]:
conf_subj_load.shape

(29, 19)

In [8]:
comp_maps_dir = os.path.join(gRAICAR_output_path, 'compMaps')
comp_maps = sorted(os.listdir(comp_maps_dir))
last_comp = conf_subj_load.shape[1]
bad_comps = [2, 3, 4, 5, 8, 9, 12] # need to change


for i in range(last_comp):
    roi_values = defaultdict(list) # ROI : list of voxel values
    roi_avgs_z_scored = {} # ROI : z-score of averaged voxel values belonging to ROI
    img = nib.load(os.path.join(comp_maps_dir, comp_maps[i]))
    data = img.get_fdata()
    for index, value in np.ndenumerate(data):
        if atlas_data[index[0], index[1], index[2]] != 0:
            roi_label = atlas_data[index[0], index[1], index[2]]
            roi_values[roi_label].append(value)
    pooled_roi_avgs = [np.mean(value) for key, value in roi_values.items()] # pool averaged voxel values for each ROI into a single list
    mean_roi_avg = np.mean(pooled_roi_avgs) # mean of pooled averaged voxel values
    std_roi_avg = np.std(pooled_roi_avgs) # standard deviation of pooled averaged voxel values
    for key, value in roi_values.items():
        roi_avgs_z_scored[key] = (np.mean(value) - mean_roi_avg) / std_roi_avg # compute z-score for each ROI
    comp_status = "Bad" if i + 1 in bad_comps else "Good"
    print(f"Prominent ROIs in Component {i+1} ({comp_status}):")
    for key, value in roi_avgs_z_scored.items():
        # if value > 2.33:
        if value > 1.64:
            print(f"[{int(key)}], {roi_names[int(key) - 1]}")

Prominent ROIs in Component 1 (Good):
[84], Infralimbic cortex (right)

[10], Brainstem_left

[48], Striatum, posterior (left)

Prominent ROIs in Component 2 (Bad):
[113], Thalamus, posterior (right)

[53], Thalamus, posterior (left)

Prominent ROIs in Component 3 (Bad):
[77], Fimbria_right

[106], Striatum, medial (right)

[112], Thalamus, anterior (right)

[78], Fornix_right

[19], Fornix_left

Prominent ROIs in Component 4 (Bad):
[91], Rest_of_Midbrain_right

[110], Superior_Colliculus_right

[70], Central_Gray_right

[33], Rest_of_Midbrain_left

[11], Central_Gray_left 

Prominent ROIs in Component 5 (Bad):
[107], Striatum, posterior (right)

Prominent ROIs in Component 6 (Good):
[107], Striatum, posterior (right)

[106], Striatum, medial (right)

[18], Fimbria_left

[47], Striatum, medial (left)

[48], Striatum, posterior (left)

Prominent ROIs in Component 7 (Good):
[78], Fornix_right

[84], Infralimbic cortex (right)

[21], Genu_corpus_callosum_left

[30], Prelimbic cortex (left

In [9]:
# Aligned components exhibiting significant group differences

# for i in range(0, last_comp):
#     r13 = conf_subj_load[:8, i]
#     SHM = conf_subj_load[8:19, i]
#     veh = conf_subj_load[19:, i]
#     data = np.concatenate([r13, SHM, veh])
#     groups = ['r13'] * len(r13) + ['SHM'] * len(SHM) + ['veh'] * len(veh)

#     F_stat, p_value = f_oneway(r13, SHM, veh) # ANOVA to determine aligned components with significant group differences
    
#     if p_value < 0.05:
#         fig, ax = plt.subplots()
#         colors = ['green', 'blue', 'red']
#         plot = ax.boxplot([r13, SHM, veh], labels = ['r13', 'SHM', 'veh'], patch_artist = True)
#         plt.xlabel('Group')
#         plt.ylabel('Confidence')
#         plt.title(f"Confidence of Subject Loads for Aligned Component {i+1}")
        
#         for patch, color in zip(plot['boxes'], colors):
#             patch.set_facecolor(color) # fill boxes with color
#         plt.show()

#         print(f'ANOVA result: F = {F_stat}, p = {p_value}')
        
#         tukey = pairwise_tukeyhsd(endog=data, groups=groups, alpha=0.05) # pairiwse Tukey test to determine which groups are actually significantly different
#         print(tukey)

#         # identify most prominent ROIs (average the voxel values for each ROI, z-score normalize averages, then threshold (
#         comp_maps_dir = '/Users/fei/Desktop/Harris_Lab/dFNC/gRAICAR/FPI_PID7/output/compMaps'
#         comp_maps = sorted(os.listdir(comp_maps_dir))
#         img = nib.load(os.path.join(comp_maps_dir, comp_maps[i]))
#         data = img.get_fdata()
#         header = img.header
#         affine = img.affine
#         flattened_data = data.flatten()
#         mean = np.mean(flattened_data)
#         std = np.std(flattened_data)
#         z_scores = (data - mean) / std # z-score normalize data

#         signif_rois = set()

#         rois = {}
#         for x in range(96):
#             for y in range(96):
#                 for z in range(25):
#                     rois[(x, y, z)] = z_scores[x, y, z] # roi coordinates: z_score ##### USE EXPLAINED VARIANCE INSTEAD
#         sorted_rois = dict(sorted(rois.items(), key=lambda items: items[1], reverse=True)) # sort rois by z_score in descending order
#         for key, value in sorted_rois.items():
#             if atlas_data[key[0], key[1], key[2]] != 0: # we want to avoid unlabeled coordinates
#                 signif_rois.add(int(atlas_data[key[0], key[1], key[2]]))
#             if len(signif_rois) == 10: # specifies how many ROIs to include, can be changed as needed
#                 signif_rois = sorted(signif_rois)
#                 break
                
#         print(f"Top {len(signif_rois)} most prominent ROIs (sorted left to right):\n")
#         for roi in signif_rois:
#             print(f"[{roi}], {roi_names[roi - 1]}")